# Fillet Weld Strength Calculator

Calculates the weld shear stress in a fillet weld from known plate direct, shear, and bending stresses. The formulas are arranged using a one-inch weld length basis.


## Formula Basis

- Fillet throat, `t_w = 0.707 L_w`
- Weld shear to plate direct or shear stress ratio, `R_q = t_w / (2 t_p)`
- Plate bending section modulus per unit length, `Z_p = (t_p^3 / 12) / (t_p / 2)`
- Plate centerline to weld throat centroid, `y = (t_p / 2 + t_w) / 2`
- Weld bending section modulus by parallel-axis approximation, `Z_w = 2 t_w y`
- Weld bending ratio, `R_b = Z_p / Z_w`
- Total weld shear stress, `T_w = sqrt((tau_wd + tau_wb)^2 + tau_wp^2)`

All dimensions in this notebook are inches. Stresses are entered and reported in psi.

In [ ]:
import math
from dataclasses import dataclass
from typing import Optional

import sympy as sp
import matplotlib.pyplot as plt
from matplotlib.patches import Arc, Circle, Polygon, Rectangle


# ---------------------------------------------------------------------------
# INPUTS -- edit values here
# ---------------------------------------------------------------------------
inputs = {
    # Job identification
    "calculated_by": "A. Trepanier",
    "job_or_moc": "Example calculation",
    "location": "Tee/web-to-plate fillet weld",

    # Geometry
    "plate_thickness": 16.0,      # t_p, in
    "weld_leg_length": 6.0,       # L_w, in

    # Plate stresses
    "plate_direct_stress": 50.0,  # sigma_d, psi
    "plate_shear_stress": 25.0,   # tau_p, psi
    "plate_bending_stress": 60.0, # sigma_b, psi

    # Optional criterion. Leave as None if only calculating demand.
    "allowable_weld_shear": 15000, # psi
}

In [ ]:
@dataclass(frozen=True)
class FilletWeldResult:
    plate_thickness: float
    weld_leg_length: float
    weld_throat: float
    direct_shear_ratio: float
    plate_section_modulus: float
    weld_centroid_y: float
    weld_section_modulus: float
    bending_shear_ratio: float
    weld_shear_from_direct: float
    weld_shear_from_plate_shear: float
    weld_shear_from_bending: float
    total_weld_shear: float
    allowable_weld_shear: Optional[float]
    margin: Optional[float]
    utilization: Optional[float]
    passes: Optional[bool]


def _require_positive(name, value):
    if value <= 0:
        raise ValueError(f"{name} must be greater than zero; got {value!r}")


def calculate_fillet_weld(case):
    """Calculate weld stresses from the input dictionary."""
    t_p = float(case["plate_thickness"])
    L_w = float(case["weld_leg_length"])
    sigma_d = float(case["plate_direct_stress"])
    tau_p = float(case["plate_shear_stress"])
    sigma_b = float(case["plate_bending_stress"])
    allowable = case.get("allowable_weld_shear")

    _require_positive("plate_thickness", t_p)
    _require_positive("weld_leg_length", L_w)

    t_w = L_w / math.sqrt(2.0)
    R_q = t_w / (2.0 * t_p)

    Z_p = (t_p**3 / 12.0) / (t_p / 2.0)
    y = (t_p / 2.0 + t_w) / 2.0
    Z_w = 2.0 * t_w * y
    R_b = Z_p / Z_w

    tau_wd = abs(sigma_d * R_q)
    tau_wp = abs(tau_p * R_q)
    tau_wb = abs(sigma_b * R_b)
    total = math.sqrt((tau_wd + tau_wb) ** 2 + tau_wp**2)

    if allowable is None:
        margin = None
        utilization = None
        passes = None
    else:
        allowable = float(allowable)
        _require_positive("allowable_weld_shear", allowable)
        margin = allowable - total
        utilization = total / allowable
        passes = total <= allowable

    return FilletWeldResult(
        plate_thickness=t_p,
        weld_leg_length=L_w,
        weld_throat=t_w,
        direct_shear_ratio=R_q,
        plate_section_modulus=Z_p,
        weld_centroid_y=y,
        weld_section_modulus=Z_w,
        bending_shear_ratio=R_b,
        weld_shear_from_direct=tau_wd,
        weld_shear_from_plate_shear=tau_wp,
        weld_shear_from_bending=tau_wb,
        total_weld_shear=total,
        allowable_weld_shear=allowable,
        margin=margin,
        utilization=utilization,
        passes=passes,
    )


def print_results(case, result):
    bar = "=" * 72
    print(bar)
    print("FILLET WELD STRENGTH CALCULATION")
    print(bar)
    print(f"  Job:        {case.get('job_or_moc', '')}  ({case.get('calculated_by', '')})")
    print(f"  Location:   {case.get('location', '')}")
    print("\nGeometry:")
    print(f"  Plate thickness, t_p                 : {result.plate_thickness:>12.3f} in")
    print(f"  Weld leg length, L_w                 : {result.weld_leg_length:>12.3f} in")
    print(f"  Weld throat, t_w = 0.707 L_w         : {result.weld_throat:>12.3f} in")
    print(f"  Plate CL to weld throat centroid, y  : {result.weld_centroid_y:>12.3f} in")
    print("\nSection properties and ratios:")
    print(f"  R_q = t_w / (2 t_p)                  : {result.direct_shear_ratio:>12.4f}")
    print(f"  Z_p                                 : {result.plate_section_modulus:>12.3f} in^2")
    print(f"  Z_w                                 : {result.weld_section_modulus:>12.3f} in^2")
    print(f"  R_b = Z_p / Z_w                      : {result.bending_shear_ratio:>12.4f}")
    print("\nWeld shear stresses:")
    print(f"  Direct stress component, tau_wd      : {result.weld_shear_from_direct:>12.3f} psi")
    print(f"  Plate shear component, tau_wp        : {result.weld_shear_from_plate_shear:>12.3f} psi")
    print(f"  Bending stress component, tau_wb     : {result.weld_shear_from_bending:>12.3f} psi")
    print(f"  Total weld shear, T_w                : {result.total_weld_shear:>12.3f} psi")

    if result.allowable_weld_shear is not None:
        verdict = "PASS" if result.passes else "FAIL"
        print("\nCriterion:")
        print(f"  Allowable weld shear                 : {result.allowable_weld_shear:>12.3f} psi")
        print(f"  Margin                               : {result.margin:>12.3f} psi")
        print(f"  Utilization                          : {result.utilization:>12.1%}")
        print(f"  Verdict                              : {verdict}")
    else:
        print("\nCriterion: no allowable weld shear entered; demand only.")
    print(bar)

## SymPy Equation Trace


In [ ]:
# ---------------------------------------------------------------------------
# SYMPY EQUATION TRACE -- clean algebra / troubleshooting
# ---------------------------------------------------------------------------
from IPython.display import Math, Markdown, display


def unit_latex(unit):
    units = {
        "in": r"\mathrm{in}",
        "in^2": r"\mathrm{in}^2",
        "psi": r"\mathrm{psi}",
    }
    return units.get(unit, unit)


def value_latex(value, unit="", ratio_digits=6):
    value = float(sp.N(value))
    if unit == "%":
        return f"{value:.2%}".replace("%", r"\%")
    if unit == "-":
        return f"{value:.{ratio_digits}f}"
    suffix = rf"\ {unit_latex(unit)}" if unit else ""
    return f"{value:.3f}{suffix}"


def show_step(label, lhs, rhs_latex, substitution_latex, value, unit=""):
    display(Math(
        rf"\begin{{aligned}}"
        rf"\text{{({label})}}\quad {sp.latex(lhs)} &= {rhs_latex} \\"
        rf"&= {substitution_latex} \\"
        rf"&= {value_latex(value, unit)}"
        rf"\end{{aligned}}"
    ))


# Symbol definitions
t_p, L_w, sigma_d, tau_p, sigma_b, T_allow = sp.symbols("t_p L_w sigma_d tau_p sigma_b T_allow", positive=True)
t_w, R_q, Z_p, y, Z_w, R_b = sp.symbols("t_w R_q Z_p y Z_w R_b", positive=True)
tau_wd, tau_wp, tau_wb, V_ip, V_oop, T_w = sp.symbols("tau_wd tau_wp tau_wb V_ip V_oop T_w", positive=True)
margin, utilization = sp.symbols("margin utilization")

values = {
    t_p: float(inputs["plate_thickness"]),
    L_w: float(inputs["weld_leg_length"]),
    sigma_d: float(inputs["plate_direct_stress"]),
    tau_p: float(inputs["plate_shear_stress"]),
    sigma_b: float(inputs["plate_bending_stress"]),
}
if inputs.get("allowable_weld_shear") is not None:
    values[T_allow] = float(inputs["allowable_weld_shear"])

steps = [
    (
        "weld throat",
        t_w,
        L_w / sp.sqrt(2),
        rf"\frac{{{sp.latex(L_w)}}}{{\sqrt{{2}}}}",
        lambda v: rf"\frac{{{value_latex(v[L_w], 'in')}}}{{\sqrt{{2}}}}",
        "in",
    ),
    (
        "direct/shear transfer ratio",
        R_q,
        t_w / (2 * t_p),
        rf"\frac{{{sp.latex(t_w)}}}{{2{sp.latex(t_p)}}}",
        lambda v: rf"\frac{{{value_latex(v[t_w], 'in')}}}{{2({value_latex(v[t_p], 'in')})}}",
        "-",
    ),
    (
        "plate bending section modulus",
        Z_p,
        (t_p**3 / 12) / (t_p / 2),
        rf"\frac{{{sp.latex(t_p)}^3/12}}{{{sp.latex(t_p)}/2}}",
        lambda v: rf"\frac{{({value_latex(v[t_p], 'in')})^3/12}}{{{value_latex(v[t_p], 'in')}/2}}",
        "in^2",
    ),
    (
        "plate CL to weld centroid",
        y,
        (t_p / 2 + t_w) / 2,
        rf"\frac{{{sp.latex(t_p)}/2 + {sp.latex(t_w)}}}{{2}}",
        lambda v: rf"\frac{{{value_latex(v[t_p], 'in')}/2 + {value_latex(v[t_w], 'in')}}}{{2}}",
        "in",
    ),
    (
        "weld bending section modulus",
        Z_w,
        2 * t_w * y,
        rf"2{sp.latex(t_w)}{sp.latex(y)}",
        lambda v: rf"2({value_latex(v[t_w], 'in')})({value_latex(v[y], 'in')})",
        "in^2",
    ),
    (
        "bending transfer ratio",
        R_b,
        Z_p / Z_w,
        rf"\frac{{{sp.latex(Z_p)}}}{{{sp.latex(Z_w)}}}",
        lambda v: rf"\frac{{{value_latex(v[Z_p], 'in^2')}}}{{{value_latex(v[Z_w], 'in^2')}}}",
        "-",
    ),
    (
        "weld shear from direct stress",
        tau_wd,
        sp.Abs(sigma_d * R_q),
        rf"\left|{sp.latex(sigma_d)}{sp.latex(R_q)}\right|",
        lambda v: rf"\left|({value_latex(v[sigma_d], 'psi')})({value_latex(v[R_q], '-')})\right|",
        "psi",
    ),
    (
        "weld shear from plate shear",
        tau_wp,
        sp.Abs(tau_p * R_q),
        rf"\left|{sp.latex(tau_p)}{sp.latex(R_q)}\right|",
        lambda v: rf"\left|({value_latex(v[tau_p], 'psi')})({value_latex(v[R_q], '-')})\right|",
        "psi",
    ),
    (
        "weld shear from bending stress",
        tau_wb,
        sp.Abs(sigma_b * R_b),
        rf"\left|{sp.latex(sigma_b)}{sp.latex(R_b)}\right|",
        lambda v: rf"\left|({value_latex(v[sigma_b], 'psi')})({value_latex(v[R_b], '-')})\right|",
        "psi",
    ),
    (
        "in-plane weld shear",
        V_ip,
        tau_wd + tau_wb,
        rf"{sp.latex(tau_wd)} + {sp.latex(tau_wb)}",
        lambda v: rf"{value_latex(v[tau_wd], 'psi')} + {value_latex(v[tau_wb], 'psi')}",
        "psi",
    ),
    (
        "out-of-plane weld shear",
        V_oop,
        tau_wp,
        sp.latex(tau_wp),
        lambda v: value_latex(v[tau_wp], "psi"),
        "psi",
    ),
    (
        "resultant weld shear",
        T_w,
        sp.sqrt(V_ip**2 + V_oop**2),
        rf"\sqrt{{{sp.latex(V_ip)}^2 + {sp.latex(V_oop)}^2}}",
        lambda v: rf"\sqrt{{({value_latex(v[V_ip], 'psi')})^2 + ({value_latex(v[V_oop], 'psi')})^2}}",
        "psi",
    ),
]
if T_allow in values:
    steps.extend([
        (
            "allowable margin",
            margin,
            T_allow - T_w,
            rf"{sp.latex(T_allow)} - {sp.latex(T_w)}",
            lambda v: rf"{value_latex(v[T_allow], 'psi')} - {value_latex(v[T_w], 'psi')}",
            "psi",
        ),
        (
            "allowable utilization",
            utilization,
            T_w / T_allow,
            rf"\frac{{{sp.latex(T_w)}}}{{{sp.latex(T_allow)}}}",
            lambda v: rf"\frac{{{value_latex(v[T_w], 'psi')}}}{{{value_latex(v[T_allow], 'psi')}}}",
            "%",
        ),
    ])


display(Markdown("### SymPy Algebra Trace"))
for label, lhs, rhs, rhs_latex, substitution_latex, unit in steps:
    result_value = sp.N(rhs.subs(values))
    show_step(label, lhs, rhs_latex, substitution_latex(values), result_value, unit)
    values[lhs] = result_value

In [ ]:
def _add_moment_symbol(ax, center, radius, color="#f58518"):
    """Add a simple 2D curved moment arrow."""
    arc = Arc(center, 2 * radius, 2 * radius, theta1=35, theta2=310, color=color, lw=2.0)
    ax.add_patch(arc)

    end_angle = math.radians(310)
    end = (center[0] + radius * math.cos(end_angle), center[1] + radius * math.sin(end_angle))
    tangent = (-math.sin(end_angle), math.cos(end_angle))
    ax.annotate(
        "",
        xy=end,
        xytext=(end[0] - 0.22 * radius * tangent[0], end[1] - 0.22 * radius * tangent[1]),
        arrowprops={"arrowstyle": "->", "color": color, "lw": 2.0},
    )
    ax.text(center[0], center[1] + 1.15 * radius, "M", color=color, fontsize=12, fontweight="bold", ha="center")


def plot_fillet_weld(case, result):
    """Draw a simple 2D weld sketch and stress table."""
    t_p = result.plate_thickness
    L_w = result.weld_leg_length
    t_w = result.weld_throat

    in_plane = result.weld_shear_from_direct + result.weld_shear_from_bending
    out_of_plane = result.weld_shear_from_plate_shear
    governing_label = "In-plane shear" if in_plane >= out_of_plane else "Out-of-plane shear"
    governing_value = max(in_plane, out_of_plane)
    governing_color = "#b22222"
    in_plane_color = governing_color if governing_label == "In-plane shear" else "#1f77b4"
    out_of_plane_color = governing_color if governing_label == "Out-of-plane shear" else "#2a7f62"

    fig, (ax, ax_stress) = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [1.35, 1.0]})

    base_width = max(3.0 * t_p, 5.2 * L_w)
    base_depth = 0.25 * t_p
    web_height = 1.25 * t_p
    web_width = 0.42 * t_p

    base = Rectangle((-base_width / 2, -base_depth), base_width, base_depth, fc="#d9d9d9", ec="#555", lw=1.2)
    web = Rectangle((-web_width / 2, 0), web_width, web_height, fc="#f2f2f2", ec="#555", lw=1.2)
    left_weld = Polygon(
        [(-web_width / 2, 0), (-web_width / 2 - L_w, 0), (-web_width / 2, L_w)],
        closed=True,
        fc="#f7c948",
        ec="#a66a00",
        lw=1.2,
    )
    right_weld = Polygon(
        [(web_width / 2, 0), (web_width / 2 + L_w, 0), (web_width / 2, L_w)],
        closed=True,
        fc="#f7c948",
        ec="#a66a00",
        lw=1.2,
    )
    for patch in (base, web, left_weld, right_weld):
        ax.add_patch(patch)

    arrow_y = web_height * 0.72
    ax.annotate(
        "",
        xy=(base_width * 0.38, arrow_y),
        xytext=(web_width / 2, arrow_y),
        arrowprops={"arrowstyle": "->", "color": in_plane_color, "lw": 2.2},
    )
    ax.text(base_width * 0.40, arrow_y, "V_ip", color=in_plane_color, fontsize=12, fontweight="bold", va="center")

    oop_center = (-web_width / 2 - 1.25 * L_w, L_w * 0.65)
    ax.add_patch(Circle(oop_center, 0.18 * L_w, fill=False, ec=out_of_plane_color, lw=2.0))
    ax.plot([oop_center[0] - 0.11 * L_w, oop_center[0] + 0.11 * L_w], [oop_center[1] - 0.11 * L_w, oop_center[1] + 0.11 * L_w], color=out_of_plane_color, lw=2.0)
    ax.plot([oop_center[0] - 0.11 * L_w, oop_center[0] + 0.11 * L_w], [oop_center[1] + 0.11 * L_w, oop_center[1] - 0.11 * L_w], color=out_of_plane_color, lw=2.0)
    ax.text(oop_center[0], oop_center[1] - 0.36 * L_w, "V_oop", color=out_of_plane_color, fontsize=12, fontweight="bold", ha="center")

    _add_moment_symbol(ax, (web_width / 2 + 1.35 * L_w, L_w * 0.95), 0.55 * L_w)
    _add_moment_symbol(ax, (-web_width / 2 - 1.35 * L_w, L_w * 0.95), 0.55 * L_w)

    ax.annotate(
        "",
        xy=(web_width / 2 + L_w, -0.08 * t_p),
        xytext=(web_width / 2, -0.08 * t_p),
        arrowprops={"arrowstyle": "<->", "color": "#333", "lw": 1.2},
    )
    ax.text(web_width / 2 + L_w / 2, -0.16 * t_p, f"L_w = {L_w:.3g} in", ha="center", va="top", fontsize=9)
    ax.text(web_width / 2 + 0.62 * L_w, 0.55 * L_w, f"t_w = {t_w:.3g} in", color="#a66a00", fontsize=9)
    ax.text(base_width / 2 * 0.72, -base_depth / 2, f"t_p = {t_p:.3g} in", color="#333", fontsize=9, va="center")

    ax.set_title("Fillet Weld Load Sketch", fontweight="bold")
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-base_width / 2 - 0.12 * base_width, base_width / 2 + 0.18 * base_width)
    ax.set_ylim(-0.40 * t_p, web_height + 0.15 * t_p)
    ax.axis("off")

    verdict = "Demand only" if result.passes is None else ("PASS" if result.passes else "FAIL")
    verdict_color = "#333" if result.passes is None else ("#1b6e1b" if result.passes else "#a40000")

    ax_stress.axis("off")
    ax_stress.set_title(f"Weld Stress Table - {verdict}", color=verdict_color, fontweight="bold")

    table_rows = [
        ["tau_wd", "Direct stress component", f"{result.weld_shear_from_direct:.1f}", "psi"],
        ["tau_wp", "Plate shear component", f"{result.weld_shear_from_plate_shear:.1f}", "psi"],
        ["tau_wb", "Bending stress component", f"{result.weld_shear_from_bending:.1f}", "psi"],
        ["V_ip", "In-plane shear", f"{in_plane:.1f}", "psi"],
        ["V_oop", "Out-of-plane shear", f"{out_of_plane:.1f}", "psi"],
        ["T_w", "Total resultant", f"{result.total_weld_shear:.1f}", "psi"],
    ]
    if result.allowable_weld_shear is not None:
        table_rows.extend([
            ["T_allow", "Allowable weld shear", f"{result.allowable_weld_shear:.1f}", "psi"],
            ["U", "Utilization", f"{result.utilization:.1%}", ""],
        ])

    table = ax_stress.table(
        cellText=table_rows,
        colLabels=["Symbol", "Stress", "Value", "Unit"],
        cellLoc="left",
        colLoc="left",
        colWidths=[0.20, 0.48, 0.22, 0.10],
        loc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9.5)
    table.scale(1.0, 1.45)

    for (row, col), cell in table.get_celld().items():
        cell.set_edgecolor("#cccccc")
        if row == 0:
            cell.set_facecolor("#eeeeee")
            cell.set_text_props(weight="bold", color="#333")
            continue
        cell.set_facecolor("#ffffff" if row % 2 else "#f8f8f8")

        symbol = table_rows[row - 1][0]
        if symbol == "tau_wb":
            cell.set_text_props(color="#f58518", weight="bold" if col == 0 else "normal")
        if symbol == "V_ip":
            cell.set_text_props(color=in_plane_color, weight="bold" if governing_label == "In-plane shear" else "normal")
        if symbol == "V_oop":
            cell.set_text_props(color=out_of_plane_color, weight="bold" if governing_label == "Out-of-plane shear" else "normal")
        if symbol == "T_w":
            cell.set_text_props(color="#333", weight="bold")

    ax_stress.text(
        0.02,
        0.06,
        f"Governing component: {governing_label} = {governing_value:.1f} psi",
        color=governing_color,
        fontsize=10.5,
        fontweight="bold",
        transform=ax_stress.transAxes,
    )

    fig.suptitle(case.get("location", "Fillet weld"), fontsize=14, fontweight="bold")
    fig.tight_layout()
    return fig


result = calculate_fillet_weld(inputs)
print_results(inputs, result)
plot_fillet_weld(inputs, result)
plt.show()